# InsightFace Face Recognition Colab

這份 notebook 會在 Google Colab 上安裝並執行 [deepinsight/insightface](https://github.com/deepinsight/insightface)，用來比對兩張照片中的臉是否可能是同一人。

流程：

1. 安裝 InsightFace 與 ONNX Runtime GPU。
2. 載入 `buffalo_l` 模型包。
3. 上傳兩張照片。
4. 每張照片取面積最大的臉。
5. 計算兩個 face embedding 的 cosine similarity。

注意：InsightFace 程式碼是 MIT License，但官方 README 說明部分訓練資料與模型僅供非商業研究用途。正式商用前請確認模型授權。

## 1. 設定 Colab Runtime 與安裝套件

在 Colab 選單選：`Runtime` -> `Change runtime type` -> `T4 GPU` 或其他 GPU。

InsightFace 0.7.3 的 Cython 模組是用 NumPy 1.x C-API 編出來的，但 Colab 預裝的 OpenCV / JAX / shap 等套件會強制要求 NumPy 2.x，兩邊衝突會出現 `np._core._multiarray_umath has no attribute _blas_supports_fpe` 之類的錯誤。

下面的安裝 cell 會：

1. 把 NumPy、OpenCV、ONNX Runtime、InsightFace 一次鎖到彼此相容的版本。
2. 安裝完成後**自動重啟 runtime**（這個重啟是必要的，否則舊的 NumPy 還在記憶體裡）。

重啟後**請從第 2 段（匯入套件）的 cell 開始往下執行，不要再執行這個安裝 cell**。

In [ ]:
!pip install -q \
    "numpy==1.26.4" \
    "opencv-python==4.10.0.84" \
    "opencv-python-headless==4.10.0.84" \
    "opencv-contrib-python==4.10.0.84" \
    "onnxruntime-gpu==1.19.2" \
    "onnx==1.16.1" \
    "insightface==0.7.3" \
    "pillow==10.4.0" \
    "scikit-image==0.24.0" \
    "scikit-learn==1.5.2" \
    "scipy==1.13.1" \
    "protobuf==4.25.5" \
    "albumentations==1.4.18" \
    "matplotlib" \
    --force-reinstall --no-cache-dir

import os

print("\n安裝完成。即將自動重啟 Colab runtime；重啟後請從第 2 段（匯入套件）的 cell 開始往下執行。")
os.kill(os.getpid(), 9)

## 2. 匯入套件並載入模型

⚠️ **這一段是 runtime 重啟之後的起點。** 上面的安裝 cell 跑完會把 runtime 殺掉，你會看到 Colab 顯示「Your session crashed」之類的訊息，那是預期行為，不是錯誤。

請從這個 cell 開始往下執行（不要再執行上面的安裝 cell）。

第一次執行 `FaceAnalysis(name="buffalo_l")` 時，InsightFace 會自動下載模型到 `~/.insightface/models/`。

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import onnxruntime as ort

from google.colab import files
from insightface.app import FaceAnalysis

print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("ONNX Runtime:", ort.__version__)

if not np.__version__.startswith("1."):
    raise RuntimeError(
        "NumPy 是 2.x，但 InsightFace 0.7.3 需要 NumPy 1.x。\n"
        "請從 Cell 1（安裝）重新跑一次（包含自動重啟），再從這裡繼續。"
    )

available_providers = ort.get_available_providers()
providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if "CUDAExecutionProvider" in available_providers else ["CPUExecutionProvider"]
ctx_id = 0 if "CUDAExecutionProvider" in providers else -1

print("ONNX Runtime providers:", available_providers)
print("Using providers:", providers)

app = FaceAnalysis(name="buffalo_l", providers=providers)
app.prepare(ctx_id=ctx_id, det_size=(640, 640))

## 3. 上傳兩張照片

執行下方 cell 後選擇兩張圖片。建議使用清楚正臉、單人照片；如果照片中有多張臉，這份 notebook 會自動選擇面積最大的臉。

In [ ]:
uploaded = files.upload()
image_paths = list(uploaded.keys())

if len(image_paths) != 2:
    raise ValueError(f"請剛好上傳兩張圖片，目前上傳了 {len(image_paths)} 張。")

image1_path, image2_path = image_paths
print("Image 1:", image1_path)
print("Image 2:", image2_path)

## 4. 定義比對工具函式

In [ ]:
def read_image(path):
    image = cv2.imread(path)
    if image is None:
        raise ValueError(f"讀不到圖片：{path}")
    return image


def face_area(face):
    x1, y1, x2, y2 = face.bbox
    return float((x2 - x1) * (y2 - y1))


def select_largest_face(faces, image_path):
    if len(faces) == 0:
        raise ValueError(f"沒有偵測到臉：{image_path}")
    return max(faces, key=face_area)


def analyze_image(path):
    image = read_image(path)
    faces = app.get(image)
    face = select_largest_face(faces, path)
    return image, face, faces


def cosine_similarity(face_a, face_b):
    return float(np.dot(face_a.normed_embedding, face_b.normed_embedding))


def draw_selected_face(image, face, title):
    output = image.copy()
    x1, y1, x2, y2 = face.bbox.astype(int)
    score = float(face.det_score)

    cv2.rectangle(output, (x1, y1), (x2, y2), (0, 255, 0), 3)
    cv2.putText(output, f"score={score:.2f}", (x1, max(y1 - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    if face.kps is not None:
        for x, y in face.kps.astype(int):
            cv2.circle(output, (x, y), 3, (255, 0, 0), -1)

    return cv2.cvtColor(output, cv2.COLOR_BGR2RGB), title

## 5. 執行臉部比對

`THRESHOLD` 是粗略參考值，不是官方固定標準。光線、角度、遮擋、照片品質都會影響分數；若要用在正式產品，應用自己的資料集校正 threshold。

In [ ]:
THRESHOLD = 0.35

image1, face1, faces1 = analyze_image(image1_path)
image2, face2, faces2 = analyze_image(image2_path)

similarity = cosine_similarity(face1, face2)
is_same_person = similarity >= THRESHOLD

print(f"{image1_path}: detected {len(faces1)} face(s), selected bbox={face1.bbox.astype(int).tolist()}")
print(f"{image2_path}: detected {len(faces2)} face(s), selected bbox={face2.bbox.astype(int).tolist()}")
print(f"Cosine similarity: {similarity:.4f}")
print(f"Threshold: {THRESHOLD:.2f}")
print("Result:", "可能是同一人" if is_same_person else "可能不是同一人")

vis1, title1 = draw_selected_face(image1, face1, image1_path)
vis2, title2 = draw_selected_face(image2, face2, image2_path)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(vis1)
plt.title(title1)
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(vis2)
plt.title(title2)
plt.axis("off")

plt.suptitle(f"Similarity={similarity:.4f} | {'Same person?' if is_same_person else 'Different person?'}")
plt.show()

## 6. 可選：手動調整判斷門檻

如果你想讓判斷更嚴格，把 threshold 調高；想讓判斷更寬鬆，把 threshold 調低。

In [ ]:
for threshold in [0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    result = "可能是同一人" if similarity >= threshold else "可能不是同一人"
    print(f"threshold={threshold:.2f}: {result}")